In [ ]:
import pandas as pd
from sklearn.preprocessing import StandardScaler
import torch
import torch.nn as nn
import torch.nn.init as init
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from tqdm.auto import trange
%matplotlib inline
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import numpy as np

In [ ]:
if torch.cuda.is_available():
    device = torch.device("cuda:0")
else:
    device = torch.device("cpu")
print("Device:", device)

### Read Basket Option Data

In [ ]:
base_filename = 'basket_option100000x10'

In [ ]:
# Basket data stored in 10 files - load and concatenate
basket_data_list = list()
for i in range(0,10):
    new_basket_data = pd.read_csv(base_filename + str(i) + '.csv', index_col=[0])
    basket_data_list.append(new_basket_data) 
    
basket_data = pd.concat(basket_data_list)
basket_data.reset_index(inplace=True)

In [ ]:
drop_list = ['errorEstimate', 'samples', 'processTime']
basket_dataset = basket_data.drop(drop_list, axis=1)
basket_dataset.head()

### Split Data

In [ ]:
train_df = basket_dataset[:-40000]
test_df = basket_dataset[-40000:-20000]
validate_df = basket_dataset[-20000:]

In [ ]:
def prepareData(df):
    y = pd.DataFrame(df['optionValue'])
    X = df.drop(['optionValue'], axis=1)
    return X, y

In [ ]:
X_train, y_train = prepareData(train_df)
X_test, y_test = prepareData(test_df)
X_val, y_val = prepareData(validate_df)

### Standard Scaling

In [ ]:
standard_scalar = StandardScaler()
X_train = standard_scalar.fit_transform(X_train)
X_test = standard_scalar.transform(X_test)
X_val = standard_scalar.transform(X_val)

### Build Model

In [ ]:
def train_model(model, train_loader, test_loader, loss_fn, optimizer, epochs):
    train_errors, test_errors = [], []
    train_rmse_errors, test_rmse_errors = [], []

    tqdm_epoch = trange(epochs)
    for epoch in tqdm_epoch:
        model.train()
        train_loss, train_sq_sum = 0.0, 0.0

        # Training
        for batch_X, batch_y in train_loader:
            # Forward pass
            outputs = model(batch_X)
            loss = loss_fn(outputs, batch_y)
            train_sq_sum += torch.sum((outputs - batch_y) ** 2).item()
            
            # Backward pass and optimization
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            train_loss += loss.item() * batch_X.size(0)

        train_loss /= len(train_loader.dataset)
        train_rmse = np.sqrt(train_sq_sum / len(train_loader.dataset))
        train_errors.append(train_loss)
        train_rmse_errors.append(train_rmse)
        
        # Evaluation on test set
        model.eval()
        test_loss, test_sq_sum = 0.0, 0.0
        with torch.no_grad():
            for batch_X, batch_y in test_loader:
                outputs = model(batch_X)
                loss = loss_fn(outputs, batch_y)
                test_sq_sum += torch.sum((outputs - batch_y) ** 2).item()
                test_loss += loss.item() * batch_X.size(0)

        test_loss /= len(test_loader.dataset)
        test_rmse = np.sqrt(test_sq_sum / len(test_loader.dataset))
        test_errors.append(test_loss)
        test_rmse_errors.append(test_rmse)

        tqdm_epoch.set_description(f"Epoch {epoch+1}/{epochs} - Train loss: {train_loss:.4f}, \
            Test loss: {test_loss:.4f}, Train RMSE: {train_rmse:.4f}, Test RMSE: {test_rmse:.4f}")

    history = dict()
    history['train_loss'] = train_errors
    history['test_loss'] = test_errors
    history['train_rmse'] = train_rmse_errors
    history['test_rmse'] = test_rmse_errors

    return history

In [ ]:
no_epochs = 50
batch_size = 2048

In [ ]:
train_x = torch.Tensor(X_train).to(device)
train_y = torch.Tensor(y_train.to_numpy().copy()).to(device)
train_dataset = TensorDataset(train_x, train_y)
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=10000, 
                                           shuffle=True, drop_last=True)

val_x = torch.Tensor(X_val).to(device)
val_y = torch.Tensor(y_val.to_numpy().copy()).to(device)
val_dataset = TensorDataset(val_x, val_y)
val_loader = torch.utils.data.DataLoader(val_dataset, batch_size=1000, 
                                          shuffle=False, drop_last=True)

In [ ]:
class FeedForwardNN(nn.Module):
    def __init__(self, no_layers, use_batch_norm=False):
        super(FeedForwardNN, self).__init__()
        self.use_batch_norm = use_batch_norm
        
        # Initializing the first hidden layer separately to handle the input size
        layers = [nn.Linear(28, 200)]
        if self.use_batch_norm:
            layers.append(nn.BatchNorm1d(200))
        
        # Appending remaining layers
        for _ in range(no_layers):
            layers.append(nn.Linear(200, 200))
            if self.use_batch_norm:
                layers.append(nn.BatchNorm1d(200))
        
        # Convert the list of layers into nn.ModuleList
        self.hidden_layers = nn.ModuleList(layers)
        self.output_layer = nn.Linear(200, 1)

    def forward(self, x):
        for layer in self.hidden_layers:
            if isinstance(layer, nn.Linear):
                x = layer(x)
                x = torch.relu(x)
            elif isinstance(layer, nn.BatchNorm1d) and self.use_batch_norm:
                x = layer(x)
        x = self.output_layer(x)
        return x

In [ ]:
history_dict = dict()
loss_fn = nn.MSELoss()
learning_rates = [0.00001, 0.0001, 0.001, 0.01, 0.1]

### Adam - No Batch Normalization

In [ ]:
for ilr in learning_rates:
    model = FeedForwardNN(5, False).to(device)
    optimizer = optim.Adam(model.parameters(), lr=ilr)
    key = 'Adam lr=' + str(ilr)
    history_dict[key] = train_model(model, train_loader, val_loader, loss_fn, optimizer, no_epochs)

In [ ]:
for ilr in learning_rates:
    model = FeedForwardNN(5, True).to(device)
    optimizer = optim.Adam(model.parameters(), lr=ilr)
    key = 'Adam + BN lr=' + str(ilr)
    history_dict[key] = train_model(model, train_loader, val_loader, loss_fn, optimizer, no_epochs)

### Adam with Batch Normalization

In [ ]:
epochs = range(1, no_epochs + 1)

### Plot Training Loss

In [ ]:
fig, ax = plt.subplots(figsize=(10,7))

from cycler import cycler
monochrome = (cycler('color', ['k']) * cycler('linestyle', ['-', '--', ':', '-.']) * cycler('marker', ['^',',', '.']))

ax.set_prop_cycle(monochrome)
for ilr in history_dict:
    train_loss_hist = history_dict[ilr]
    train_loss_values = train_loss_hist['train_loss']
    ax.plot(epochs, train_loss_values, label = ilr)

ax.set_xlabel('Epochs')
ax.set_ylabel('Training Loss')
ax.set_ylim(0, 3)
ax.legend()
fig.savefig('TestBasketBatchNorm.png', dpi=300, bbox_inches='tight')

### Plot Validation Loss

In [ ]:
def smooth_curve(points, factor=0.9):
    smoothed_points = []
    for point in points:
        if smoothed_points:
            previous = smoothed_points[-1]
            smoothed_points.append(previous * factor + point * (1 - factor))
        else:
            smoothed_points.append(point)
    return smoothed_points

In [ ]:
fig, ax = plt.subplots(figsize=(10,7))

from cycler import cycler
monochrome = (cycler('color', ['k']) * cycler('linestyle', ['-', '--', ':', '-.']) * cycler('marker', ['^',',', '.']))

ax.set_prop_cycle(monochrome)
for ilr in history_dict:
    val_loss_hist = history_dict[ilr]
    val_loss_values = smooth_curve(val_loss_hist['test_loss'])
    ax.plot(epochs, val_loss_values, label = ilr)

ax.set_xlabel('Epochs')
ax.set_ylabel('Validation Loss')
ax.set_ylim(0, 7)
ax.legend(loc='center left', bbox_to_anchor=(1, 0.5))
fig.savefig('TestBasketBatchNormVal.png', dpi=300, bbox_inches='tight')